In [ ]:

import os
import urllib.parse
import prettytable
from dotenv import load_dotenv

load_dotenv(os.path.join("..", ".env"))

SERVER   = os.getenv("DB_SERVER")
DATABASE = os.getenv("DB_NAME", "OlistDB")
USER     = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
DRIVER   = os.getenv("DB_DRIVER", "ODBC Driver 17 for SQL Server")


missing = [k for k, v in {"SERVER": SERVER, "USER": USER, "PASSWORD": PASSWORD}.items() if not v]
if missing:
    raise EnvironmentError(f"❌ Variáveis ausentes no .env: {', '.join(missing)}")

print(f"📦 Servidor : {SERVER}")
print(f"📦 Banco    : {DATABASE}")
print(f"📦 Usuário  : {USER}")


_available_styles = [k for k in prettytable.__dict__ if k.isupper() and isinstance(prettytable.__dict__[k], int)]
_preferred = ["DEFAULT", "SINGLE_BORDER", "MARKDOWN", "PLAIN_COLUMNS"]
PRETTY_STYLE = next((s for s in _preferred if s in _available_styles), _available_styles[0] if _available_styles else "DEFAULT")

print(f"🎨 PrettyTable style: {PRETTY_STYLE}  (disponíveis: {_available_styles})")

odbc_str = (
    f"DRIVER={{{DRIVER}}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    f"UID={USER};"
    f"PWD={PASSWORD};"
    f"Encrypt=yes;"
    f"TrustServerCertificate=yes;"
    f"MARS_Connection=yes;"
)

connection_url = f"mssql+pyodbc:///?odbc_connect={urllib.parse.quote_plus(odbc_str)}"


%reload_ext sql
%config SqlMagic.feedback    = True         # Exibe quantidade de linhas retornadas
%config SqlMagic.autopandas  = False        # Retorna ResultSet (mais leve)
%config SqlMagic.displaycon  = False        # Oculta string de conexão no output
%config SqlMagic.style       = PRETTY_STYLE # Fix: usa estilo detectado dinamicamente


%sql {connection_url}

print("\n✅ Conexão estabelecida com OlistDB!")
print("   Use %%sql em uma célula separada para executar queries.")

📦 Servidor : LUCAS
📦 Banco    : OlistDB
📦 Usuário  : sa
🎨 PrettyTable style: DEFAULT  (disponíveis: ['ALL', 'DEFAULT', 'DOUBLE_BORDER', 'FRAME', 'HEADER', 'MARKDOWN', 'MSWORD_FRIENDLY', 'NONE', 'ORGMODE', 'PLAIN_COLUMNS', 'RANDOM', 'SINGLE_BORDER'])

✅ Conexão estabelecida com OlistDB!
   Use %%sql em uma célula separada para executar queries.


## Modelo de Segmentação RFM

### Camada de Ingestão e Fundação (customers, orders, order_items)

O que faz: Consome os dados limpos e modelados diretamente do Star Schema físico (dim_customers, fct_orders, fct_order_items).

Racional: Garante a governança de dados. O modelo analítico nunca lê dados brutos (raw), protegendo a linhagem e eliminando registros duplicados antes de iniciar os cálculos.

In [11]:
%%sql 

SELECT TOP 10 * FROM analytics.dim_customers

Done.


customer_sk,customer_id,customer_unique_id,zip_code_prefix,city,state
2cb88cd661bfafef57b9050432d616d5,24417cdc12fe255a7cdd87ff914a9af8,9b4bb9d14ad112db54cc539ab2b8fd05,3581,sao paulo,SP
68da698d40e36c6382b7966da33303bc,2441a257e681c06ff2101ed765647ada,a27b4653c3c45bbb1b1c9f12c91e6c9f,4116,sao paulo,SP
a9d3aa464a73422de44607df3800cd5f,2441cecb43e28589bf4c3703ae8d6980,b778dfee1fc0527d3dc14c5a0631bd2a,4534,sao paulo,SP
757d8aa0668c4c6091665e1281178e58,24436139c80a4ae51c0e3a98469ed7d4,b2c2e2635aded82494c5be5843664b74,3690,sao paulo,SP
022e2f62c465fca33dce131df3c8a429,245218ed5faf26b2117f4edfb033ec80,59775d57770e03aa8ad34d4c54528994,1257,sao paulo,SP
fbff1e4205c3eed21ef4d0689db8edd9,24581467d0bfc119d33575ece54987b2,6629514c79eec8eab37b50223c771c7e,5782,sao paulo,SP
0c4acdc84b8e7c287ea0aae3e25e74b4,245d34e7c0f12c871d9cf7941a41fc9c,898f9e5d271975c22949b9693239bd8b,5164,sao paulo,SP
894aa66179fd2a2880e3674cbb8d1f50,2467f4b55c35ab6b015f09d0312570a3,0d8696c6a5acf0374e17b9a0a59e1296,5057,sao paulo,SP
dd06225cde331df3a8beeadba000ba6b,2472db5dcd1737948d7c75037c8d9ebb,99eb34555f2b022b5c8cf242b9802e78,3089,sao paulo,SP
db112eb4cff6fcc60153100e5a81e355,247eb12909a8db3cc579b1260c3061e6,ceb4a58a238175053bbbf54ca503de8c,8485,sao paulo,SP


In [24]:
%%sql 

SELECT TOP 10 
    customer_sk,
    order_id,
    purchase_at 
FROM analytics.fct_orders

Done.


customer_sk,order_id,purchase_at
4454477b71dbc9663a4322cd3a85f33b,00048cc3ae777c65dbb7d2a0634bc1ea,2017-05-15 21:42:34
db5b63fec11aef2c25b1d5af3ea55007,00169e31ef4b29deaae414f9a5e95929,2018-01-16 09:26:39
84b8e31fe1db35dd13d2386191911eef,00cb18db3d9c03f8386e7fc70cad2504,2017-07-12 00:06:51
4b9269ebe6fd79a0ccafca887a6a891a,228f12f010b1363cba76e442bd378e53,2017-03-09 15:37:50
2128b579745ad59e259ea7e07148c836,06b2c7035561ef12b16045ef4bf459e2,2017-07-05 07:54:14
bcd78976ed2016abae624355d9341e73,06be8b2c2a6e818814aa5416c3512ca3,2018-07-23 22:05:53
64532468c4aa2be402e087b665c15dad,06c5eb90406de0ba873e721e9182ecfd,2018-06-12 08:46:10
933cd1b617c2980397a25e1310e5e228,06ca213ef85421d96ee0fe529d0f5885,2017-07-11 16:36:24
c3e6e7cf2149919af0289b700f96cf17,06ff6cb289c9f1d1ccacc6764874e3c2,2017-08-15 22:42:16
ff72a677857f41e619aff101fb774090,07629443859f99dfac4443b102f31ead,2018-08-20 18:55:21


In [21]:
%%sql 
SELECT top 10
        order_sk,
        SUM(total_item_amount) AS total_order_amount
    FROM analytics.fct_order_items
    GROUP BY order_sk

Done.


order_sk,total_order_amount
069959480cc0beed4dc0b27c37db91e4,97.32
4f95254acf31bbe0bbed397b42bedaff,310.21
96d0f84086d5e3fbc1bdc96eba14c862,180.94
3c368b8fb181dcb37611e29928fd1b2e,102.15
d375b5a873221996e2e2facc397a93c0,76.11
43ad432896013c5d1a5aab707fd167e8,121.70
f61febec0d307632927a78dfa74a1286,213.75
308394facc42e76771c0940953a898e9,35.71
290779a0c614fa98db0a3817ef445a41,125.04
a486b77a4c3574f75fb90452393a71e7,100.75


### Consolidação do Comportamento (customer_behavior)

O que faz: Agrupa o histórico de compras por cliente único (customer_unique_id) para extrair os três pilares brutos do comportamento de consumo:
- A data da última compra realizada.
- A contagem de pedidos distintos.
- A soma do valor acumulado dos itens comprados.

In [18]:
%%sql

WITH customers AS (
    SELECT * FROM analytics.dim_customers
),

orders AS (
    SELECT * FROM analytics.fct_orders
),

order_items AS (
    SELECT 
        order_sk,
        SUM(total_item_amount) AS total_order_amount
    FROM analytics.fct_order_items
    GROUP BY order_sk
),

customer_behavior AS (
    SELECT
        c.customer_unique_id,
        MAX(o.purchase_at) AS last_purchase_at,              -- Última compra (Recência)
        COUNT(DISTINCT o.order_id) AS total_orders,          -- Total de pedidos (Frequência)
        SUM(COALESCE(i.total_order_amount, 0)) AS total_spent -- Total gasto (Valor Monetário)
    FROM customers c
    INNER JOIN orders o ON c.customer_sk = o.customer_sk
    LEFT JOIN order_items i ON o.order_sk = i.order_sk
    GROUP BY c.customer_unique_id
)

select top 10 * from customer_behavior
ORDER BY total_spent DESC

Done.


customer_unique_id,last_purchase_at,total_orders,total_spent
0a0a92112bd4c708ca5fde585afaa872,2017-09-29 15:24:52,1,13664.08
da122df9eeddfedc1dc1f5349a1a690c,2017-04-01 15:58:41,2,7571.63
763c8b1c9c68a0229c42c9fc6f662b93,2018-07-15 14:49:44,1,7274.88
dc4802a71eae9be1dd28f5d788ceb526,2017-02-12 20:37:36,1,6929.31
459bef486812aa25204be022145caa62,2018-07-25 18:10:17,1,6922.21
ff4159b92c40ebe40454e3e6a7c35ed6,2017-05-24 18:14:34,1,6726.66
4007669dec559734d6f53e029e360987,2017-11-24 11:03:35,1,6081.54
5d0a2980b292d049061542014e8960bf,2018-07-12 12:08:36,1,4809.44
eebb5dda148d3893cdaf5b5ca3040ccb,2017-04-18 18:50:13,1,4764.34
48e1ac109decbb87765a3eade6854098,2018-06-22 12:23:19,1,4681.78


### Cálculo das Métricas Brutas (rfm_raw)

O que faz: Transforma as agregações em métricas temporais e financeiras palpáveis:
- Recência (Days): Subtrai a data da última compra do cliente da data máxima registrada no banco (ajuste dinâmico de linha de base temporal).
- Frequência: Quantidade total de conversões.
- Monetário: Lifetime Value (LTV) bruto do cliente.

In [ ]:
%%sql

WITH customers AS (
    SELECT * FROM analytics.dim_customers
),

orders AS (
    SELECT * FROM analytics.fct_orders
),

order_items AS (
    SELECT 
        order_sk,
        SUM(total_item_amount) AS total_order_amount
    FROM analytics.fct_order_items
    GROUP BY order_sk
),

customer_behavior AS (
    SELECT
        c.customer_unique_id,
        MAX(o.purchase_at) AS last_purchase_at,              -- Última compra (Recência)
        COUNT(DISTINCT o.order_id) AS total_orders,          -- Total de pedidos (Frequência)
        SUM(COALESCE(i.total_order_amount, 0)) AS total_spent -- Total gasto (Valor Monetário)
    FROM customers c
    INNER JOIN orders o ON c.customer_sk = o.customer_sk
    LEFT JOIN order_items i ON o.order_sk = i.order_sk
    GROUP BY c.customer_unique_id
),

rfm_raw AS (
    SELECT
        customer_unique_id,
        DATEDIFF(day, last_purchase_at, (SELECT MAX(purchase_at) FROM orders)) AS recency_days,
        total_orders AS frequency,
        total_spent AS monetary
    FROM customer_behavior
)

SELECT TOP 10 * FROM rfm_raw;

Done.


customer_unique_id,recency_days,frequency,monetary
dc09df53d23ae01d6b42b547913aa2ba,91,1,37.23
22ab55c824c15accbaddc8e60964255f,534,1,101.14
3efb9846fb5c603fdbe933c838c6f310,188,1,75.87
d8873ca8e3847bac684fe59564b35bdc,113,1,219.49
f134af57f6907fc43fc08417e0f056ff,328,1,136.98
48e8d93c6ed6a8df94e24f4cd188d154,155,1,158.88
2afd8407e92dbb672561a38211ee563a,163,1,291.57
79ae72fd0b476edd6a6d4c7ebf329b4c,88,1,110.58
ba7e510938ce1546511bd31efb4d5335,189,1,98.88
04fc5f7b626b7d8082d4a81ec33f8b83,380,1,82.73


### O Primeiro Modelo Estatístico (rfm_scores v1)

- O que faz: Aplica a função matemática NTILE(5) para os três pilares, gerando notas automáticas de 1 a 5 por quintis estatísticos.
- Premissa Inicial: Assumiu-se que o banco distribuiria as notas de forma equilibrada para Recência, Frequência e Valor.

In [29]:
%%sql

WITH customers AS (
    SELECT * FROM analytics.dim_customers
),

orders AS (
    SELECT * FROM analytics.fct_orders
),

order_items AS (
    SELECT 
        order_sk,
        SUM(total_item_amount) AS total_order_amount
    FROM analytics.fct_order_items
    GROUP BY order_sk
),

customer_behavior AS (
    SELECT
        c.customer_unique_id,
        MAX(o.purchase_at) AS last_purchase_at,              -- Última compra (Recência)
        COUNT(DISTINCT o.order_id) AS total_orders,          -- Total de pedidos (Frequência)
        SUM(COALESCE(i.total_order_amount, 0)) AS total_spent -- Total gasto (Valor Monetário)
    FROM customers c
    INNER JOIN orders o ON c.customer_sk = o.customer_sk
    LEFT JOIN order_items i ON o.order_sk = i.order_sk
    GROUP BY c.customer_unique_id
),

rfm_raw AS (
    SELECT
        customer_unique_id,
        DATEDIFF(day, last_purchase_at, (SELECT MAX(purchase_at) FROM orders)) AS recency_days,
        total_orders AS frequency,
        total_spent AS monetary
    FROM customer_behavior
),

rfm_scores AS (
    SELECT
        customer_unique_id,
        recency_days,
        frequency,
        monetary,
        -- Cria notas de 1 a 5 para cada métrica
        NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
        NTILE(5) OVER (ORDER BY frequency ASC) AS f_score,
        NTILE(5) OVER (ORDER BY monetary ASC) AS m_score
    FROM rfm_raw
)

-- Para testar essa terceira parte:
SELECT top 10 * FROM rfm_scores;

Done.


customer_unique_id,recency_days,frequency,monetary,r_score,f_score,m_score
54dcd10897e9d1cbb96e4eb8ed26749b,163,1,0.00,4,3,1
b5b985a49b13ddfe43dc657134e7ff94,97,1,0.00,5,2,1
80aff285a3aeb1162d1431383afd871a,96,1,0.00,5,2,1
d280c55bb8f45e367a7f24151bc99fdd,97,1,0.00,5,2,1
ca072da22ab6c6a14608c9d12c00595c,97,1,0.00,5,2,1
161e5b2b4513edb336986a8909218365,345,1,0.00,2,5,1
3ba7980dfa14cd3508fa58c4198ce202,401,1,0.00,2,3,1
a8797524cfb4b5c6f1f780cd11d51739,330,1,0.00,2,5,1
d0c00b017de45d1db2d7259f5ae2e340,454,1,0.00,1,3,1
921d58c81068176026975777ae295c05,405,1,0.00,2,3,1


### A Primeira Matriz de Negócio

- O que faz: Cria uma lógica de CASE WHEN para agrupar os clientes.
- O Ponto Cego: Esta primeira versão criou um limbo chamado 'Clientes Regulares' e regras muito rígidas de AND, empurrando perfis muito mistos para uma categoria genérica.

In [32]:
%%sql
WITH customers AS (
    SELECT * FROM analytics.dim_customers
),

orders AS (
    SELECT * FROM analytics.fct_orders
),

order_items AS (
    SELECT 
        order_sk,
        SUM(total_item_amount) AS total_order_amount
    FROM analytics.fct_order_items
    GROUP BY order_sk
),

customer_behavior AS (
    SELECT
        c.customer_unique_id,
        MAX(o.purchase_at) AS last_purchase_at,              -- Última compra (Recência)
        COUNT(DISTINCT o.order_id) AS total_orders,          -- Total de pedidos (Frequência)
        SUM(COALESCE(i.total_order_amount, 0)) AS total_spent -- Total gasto (Valor Monetário)
    FROM customers c
    INNER JOIN orders o ON c.customer_sk = o.customer_sk
    LEFT JOIN order_items i ON o.order_sk = i.order_sk
    GROUP BY c.customer_unique_id
),

rfm_raw AS (
    SELECT
        customer_unique_id,
        DATEDIFF(day, last_purchase_at, (SELECT MAX(purchase_at) FROM orders)) AS recency_days,
        total_orders AS frequency,
        total_spent AS monetary
    from customer_behavior
),

rfm_scores AS (
    SELECT
        customer_unique_id,
        recency_days,
        frequency,
        monetary,
        -- Cria notas de 1 a 5 para cada métrica
        NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
        NTILE(5) OVER (ORDER BY frequency ASC) AS f_score,
        NTILE(5) OVER (ORDER BY monetary ASC) AS m_score
    FROM rfm_raw
),

final AS (
    SELECT
        customer_unique_id,
        recency_days,
        frequency,
        monetary,
        r_score,
        f_score,
        m_score,
        -- Calcula a média das três notas
        CAST((r_score + f_score + m_score) / 3.0 AS DECIMAL(10,2)) AS rfm_score_average,
        -- Regras de negócio baseadas nas notas (CORRIGIDO AQUI)
        CASE
            -- 1. Campeões (VIP): Compraram há pouquíssimo tempo, compram muito e gastam muito
            WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'Campeões (Clientes VIP)'

            -- 2. Clientes Leais: Comportamento consistente de compra e valor em frequência média/alta
            WHEN r_score >= 3 AND f_score >= 3 AND m_score >= 3 THEN 'Clientes Leais'

            -- 3. Potenciais Leais: Compraram recentemente, têm boa frequência e ticket médio OK
            WHEN r_score >= 4 AND f_score >= 2 AND m_score >= 2 THEN 'Potenciais Leais'

            -- 4. Novos Clientes: Compras muito recentes, mas frequência ainda é a primeira
            WHEN r_score >= 4 AND f_score = 1 THEN 'Novos Clientes'

            -- 5. Não Podemos Perder (Alerta Vermelho): Clientes que gastavam muito e compravam muito, mas abandonaram a loja (R baixa)
            WHEN r_score <= 2 AND f_score >= 4 AND m_score >= 4 THEN 'Não Podemos Perder'

            -- 6. Clientes em Risco: Compravam acima da média (F/M), mas estão há muito tempo sem dar sinal de vida
            WHEN r_score <= 2 AND f_score >= 2 AND m_score >= 2 THEN 'Clientes em Risco'

            -- 7. Precisam de Atenção: Clientes medianos em tudo (nem novos, nem perdidos)
            WHEN r_score = 3 AND f_score >= 2 AND m_score >= 2 THEN 'Precisam de Atenção'

            -- 8. Prestes a Dormir: Recência e frequência já estão abaixo da média geral
            WHEN r_score = 2 AND f_score <= 2 AND m_score <= 2 THEN 'Prestes a Dormir'

            -- 9. Hibernando: Clientes que compraram pouquíssimo, gastaram pouco e a última compra já faz tempo
            WHEN r_score = 2 AND f_score <= 2 THEN 'Hibernando'

            -- 10. Perdidos / Inativos: O resto do funil
            ELSE 'Clientes Perdidos / Inativos'
        END AS customer_segmentation
    FROM rfm_scores
)

SELECT TOP 10 * FROM final;

Done.


customer_unique_id,recency_days,frequency,monetary,r_score,f_score,m_score,rfm_score_average,customer_segmentation
54dcd10897e9d1cbb96e4eb8ed26749b,163,1,0.00,4,3,1,2.67,Clientes Perdidos / Inativos
b5b985a49b13ddfe43dc657134e7ff94,97,1,0.00,5,2,1,2.67,Clientes Perdidos / Inativos
80aff285a3aeb1162d1431383afd871a,96,1,0.00,5,2,1,2.67,Clientes Perdidos / Inativos
d280c55bb8f45e367a7f24151bc99fdd,97,1,0.00,5,2,1,2.67,Clientes Perdidos / Inativos
ca072da22ab6c6a14608c9d12c00595c,97,1,0.00,5,2,1,2.67,Clientes Perdidos / Inativos
161e5b2b4513edb336986a8909218365,345,1,0.00,2,5,1,2.67,Clientes Perdidos / Inativos
3ba7980dfa14cd3508fa58c4198ce202,401,1,0.00,2,3,1,2.00,Clientes Perdidos / Inativos
a8797524cfb4b5c6f1f780cd11d51739,330,1,0.00,2,5,1,2.67,Clientes Perdidos / Inativos
d0c00b017de45d1db2d7259f5ae2e340,454,1,0.00,1,3,1,1.67,Clientes Perdidos / Inativos
921d58c81068176026975777ae295c05,405,1,0.00,2,3,1,2.00,Clientes Perdidos / Inativos


### A Descoberta do Ponto Cego (ROLLUP Geral)

- O que faz: Agrupa o resultado da primeira matriz em uma tabela consolidada trazendo volumetria, percentual e as médias de comportamento de cada grupo.

- Descoberta: O resultado trouxe quase 7.000 clientes como "Campeões VIP", mas a média de frequência deles era de apenas 1.18. Ao analisar a média global do e-commerce (1.03), descobrimos o comportamento de Long Tail (Cauda Longa): 97% da base comprou apenas 1 vez.

- O Diagnóstico: O NTILE(5) na Frequência estava "sorteando" e forçando clientes de 1 única compra a ganharem notas 4 e 5 apenas para preencher os baldes estatísticos vazios, inflando falsamente o número de VIPs do negócio.

In [34]:
%%sql
WITH customers AS (
    SELECT * FROM analytics.dim_customers
),

orders AS (
    SELECT * FROM analytics.fct_orders
),

order_items AS (
    SELECT 
        order_sk,
        SUM(total_item_amount) AS total_order_amount
    FROM analytics.fct_order_items
    GROUP BY order_sk
),

customer_behavior AS (
    SELECT
        c.customer_unique_id,
        MAX(o.purchase_at) AS last_purchase_at,
        COUNT(DISTINCT o.order_id) AS total_orders,
        SUM(COALESCE(i.total_order_amount, 0)) AS total_spent
    FROM customers c
    INNER JOIN orders o ON c.customer_sk = o.customer_sk
    LEFT JOIN order_items i ON o.order_sk = i.order_sk
    GROUP BY c.customer_unique_id
),

rfm_raw AS (
    SELECT
        customer_unique_id,
        DATEDIFF(day, last_purchase_at, (SELECT MAX(purchase_at) FROM orders)) AS recency_days,
        total_orders AS frequency,
        total_spent AS monetary
    FROM customer_behavior
),

rfm_scores AS (
    SELECT
        customer_unique_id,
        recency_days,
        frequency,
        monetary,
        NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
        NTILE(5) OVER (ORDER BY frequency ASC) AS f_score,
        NTILE(5) OVER (ORDER BY monetary ASC) AS m_score
    FROM rfm_raw
),

final AS (
    SELECT
        customer_unique_id,
        recency_days,
        frequency,
        monetary,
        r_score,
        f_score,
        m_score,
        CAST((r_score + f_score + m_score) / 3.0 AS DECIMAL(10,2)) AS rfm_score_average,
        CASE
            WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'Campeões (Clientes VIP)'
            WHEN r_score >= 3 AND f_score >= 3 AND m_score >= 3 THEN 'Clientes Leais'
            WHEN r_score >= 4 AND f_score >= 2 AND m_score >= 2 THEN 'Potenciais Leais'
            WHEN r_score >= 4 AND f_score = 1 THEN 'Novos Clientes'
            WHEN r_score <= 2 AND f_score >= 4 AND m_score >= 4 THEN 'Não Podemos Perder'
            WHEN r_score <= 2 AND f_score >= 2 AND m_score >= 2 THEN 'Clientes em Risco'
            WHEN r_score = 3 AND f_score >= 2 AND m_score >= 2 THEN 'Precisam de Atenção'
            WHEN r_score = 2 AND f_score <= 2 AND m_score <= 2 THEN 'Prestes a Dormir'
            WHEN r_score = 2 AND f_score <= 2 THEN 'Hibernando'
            ELSE 'Clientes Perdidos / Inativos'
        END AS customer_segmentation
    FROM rfm_scores
)

SELECT
    -- COALESCE substitui o NULL gerado pelo ROLLUP por um texto limpo de Total
    COALESCE(customer_segmentation, '--- TOTAL GERAL ---') AS customer_segmentation,
    COUNT(*) AS total_clientes,
    
    -- Coluna de Percentual do Total de Clientes
    CAST(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM final) AS DECIMAL(10,2)) AS percentual_da_base,
    
    -- Médias de comportamento (No total geral, trará a média global do e-commerce)
    CAST(AVG(CAST(recency_days AS DECIMAL(10,2))) AS DECIMAL(10,1)) AS media_recencia_dias,
    CAST(AVG(CAST(frequency AS DECIMAL(10,2))) AS DECIMAL(10,2)) AS media_frequencia,
    CAST(AVG(monetary) AS DECIMAL(10,2)) AS media_faturamento_por_cliente
FROM final
-- O ROLLUP força o agrupamento tradicional e cria a linha de fechamento ao final
GROUP BY ROLLUP(customer_segmentation)
-- Esse CASE garante que o TOTAL GERAL fique sempre preso na última linha da tabela
ORDER BY CASE WHEN customer_segmentation IS NULL THEN 1 ELSE 0 END, total_clientes DESC;

Done.


customer_segmentation,total_clientes,percentual_da_base,media_recencia_dias,media_frequencia,media_faturamento_por_cliente
Clientes Perdidos / Inativos,21087,21.94,314.1,1.00,74.83
Clientes em Risco,18009,18.74,449.6,1.01,153.08
Clientes Leais,14113,14.69,199.3,1.06,203.95
Potenciais Leais,10242,10.66,137.8,1.01,141.23
Novos Clientes,7519,7.82,144.9,1.00,165.72
Campeões (Clientes VIP),6970,7.25,140.2,1.18,313.05
Não Podemos Perder,6725,7.00,452.7,1.13,313.70
Precisam de Atenção,5534,5.76,274.1,1.01,139.60
Hibernando,2959,3.08,358.6,1.00,245.57
Prestes a Dormir,2938,3.06,358.5,1.00,50.06


### Refatoração e Blindagem

- O que faz: Corrige o SQL substituindo o NTILE da Frequência por uma Regra Fixa (Hardcoded):
    - 1 compra = Nota 1 (Realidade de mercado)
    - 2 compras = Nota 3 (Acima da média)
    - 3 ou mais compras = Nota 5 (Fidelidade máxima)
- Resultado: Os falsos VIPs foram eliminados (caindo de ~7.000 para 130 clientes legítimos), os "Clientes Regulares" foram pulverizados em 10 novos segmentos estratégicos (como Não Podemos Perder e Prestes a Dormir), refletindo com precisão cirúrgica a saúde do e-commerce.

In [ ]:
%%sql
WITH customers AS (
    SELECT * FROM analytics.dim_customers
),

orders AS (
    SELECT * FROM analytics.fct_orders
),

order_items AS (
    SELECT 
        order_sk,
        SUM(total_item_amount) AS total_order_amount
    FROM analytics.fct_order_items
    GROUP BY order_sk
),

customer_behavior AS (
    SELECT
        c.customer_unique_id,
        MAX(o.purchase_at) AS last_purchase_at,
        COUNT(DISTINCT o.order_id) AS total_orders,
        SUM(COALESCE(i.total_order_amount, 0)) AS total_spent
    FROM customers c
    INNER JOIN orders o ON c.customer_sk = o.customer_sk
    LEFT JOIN order_items i ON o.order_sk = i.order_sk
    GROUP BY c.customer_unique_id
),

rfm_raw AS (
    SELECT
        customer_unique_id,
        DATEDIFF(day, last_purchase_at, (SELECT MAX(purchase_at) FROM orders)) AS recency_days,
        total_orders AS frequency, -- Aqui o nome virou 'frequency'
        total_spent AS monetary
    FROM customer_behavior
),

rfm_scores AS (
    SELECT
        customer_unique_id,
        recency_days,
        frequency,
        monetary,
        NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
        CASE 
            WHEN frequency = 1 THEN 1
            WHEN frequency = 2 THEN 3
            ELSE 5
        END AS f_score,
        
        NTILE(5) OVER (ORDER BY monetary ASC) AS m_score
    FROM rfm_raw
),

final AS (
    SELECT
        customer_unique_id,
        recency_days,
        frequency,
        monetary,
        r_score,
        f_score,
        m_score,
        CAST((r_score + f_score + m_score) / 3.0 AS DECIMAL(10,2)) AS rfm_score_average,
        CASE
            WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'Campeões (Clientes VIP)'
            WHEN r_score >= 3 AND f_score >= 3 AND m_score >= 3 THEN 'Clientes Leais'
            WHEN r_score >= 4 AND f_score >= 2 AND m_score >= 2 THEN 'Potenciais Leais'
            WHEN r_score >= 4 AND f_score = 1 THEN 'Novos Clientes'
            WHEN r_score <= 2 AND f_score >= 4 AND m_score >= 4 THEN 'Não Podemos Perder'
            WHEN r_score <= 2 AND f_score >= 2 AND m_score >= 2 THEN 'Clientes em Risco'
            WHEN r_score = 3 AND f_score >= 2 AND m_score >= 2 THEN 'Precisam de Atenção'
            WHEN r_score = 2 AND f_score <= 2 AND m_score <= 2 THEN 'Prestes a Dormir'
            WHEN r_score = 2 AND f_score <= 2 THEN 'Hibernando'
            ELSE 'Clientes Perdidos / Inativos'
        END AS customer_segmentation
    FROM rfm_scores
)

SELECT
    COALESCE(customer_segmentation, '--- TOTAL GERAL ---') AS customer_segmentation,
    COUNT(*) AS total_clientes,
    CAST(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM final) AS DECIMAL(10,2)) AS percentual_da_base,
    CAST(AVG(CAST(recency_days AS DECIMAL(10,2))) AS DECIMAL(10,1)) AS media_recencia_dias,
    CAST(AVG(CAST(frequency AS DECIMAL(10,2))) AS DECIMAL(10,2)) AS media_frequencia,
    CAST(AVG(monetary) AS DECIMAL(10,2)) AS media_faturamento_por_cliente
FROM final
GROUP BY ROLLUP(customer_segmentation)
ORDER BY CASE WHEN customer_segmentation IS NULL THEN 1 ELSE 0 END, total_clientes DESC;

Done.


customer_segmentation,total_clientes,percentual_da_base,media_recencia_dias,media_frequencia,media_faturamento_por_cliente
Clientes Perdidos / Inativos,37387,38.91,397.9,1.00,154.61
Novos Clientes,37132,38.64,139.8,1.00,164.65
Hibernando,10807,11.25,366.3,1.00,241.00
Prestes a Dormir,7822,8.14,366.6,1.00,54.27
Clientes Leais,1686,1.75,184.0,2.04,312.63
Clientes em Risco,965,1.00,432.5,2.01,291.00
Campeões (Clientes VIP),130,0.14,138.9,3.52,573.47
Potenciais Leais,72,0.07,124.3,2.01,72.74
Não Podemos Perder,65,0.07,426.0,3.09,467.93
Precisam de Atenção,30,0.03,266.1,2.00,73.81
